# OCHA oPt demolition data: weekly scraper

Downloads the full `_vw_Incidents` table from the public OCHA oPt demolition dashboard and saves it as **`incidents.csv`**.

**How to use (about two minutes):**
1. Run the cells from top to bottom (Runtime > Run all).
2. When it finishes, `incidents.csv` downloads to your computer. (If it does not, open the folder icon on the left and download it from there.)
3. Check the summary printed at the end. If it says **OK**, upload `incidents.csv` to your GitHub repo, replacing the old one, next to `index.html`. The site updates itself.

Nothing here touches GitHub. If anything looks wrong, just do not upload the file.

In [ ]:
# 1. Setup
import csv, datetime as dt, sys, time, uuid, os
import requests

In [ ]:
HOST = "https://wabi-north-europe-j-primary-api.analysis.windows.net"
RESOURCE_KEY = "aae77805-0842-43a2-9d2e-aa21d43557e6"
DATASET_ID = "662c6074-e09a-484b-b964-88f284c6da35"
REPORT_ID = "b3d9a13b-65a6-42e2-884f-f1164babff32"
MODEL_ID = 1272290
ENTITY = "_vw_Incidents"

# Fallback if schema discovery fails: add the real column names here.
COLUMNS = [
    "Agriculture", "Inhabited residential", "Uninhabited residential",
    "WASH", "Infrastructure", "Livelihood", "Other",
]

HEADERS = {
    "Accept": "application/json, text/plain, */*",
    "Content-Type": "application/json;charset=UTF-8",
    "Origin": "https://app.powerbi.com",
    "Referer": "https://app.powerbi.com/",
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 "
                  "(KHTML, like Gecko) Chrome/150.0.0.0 Safari/537.36",
    "X-PowerBI-ResourceKey": RESOURCE_KEY,
}

session = requests.Session()
session.headers.update(HEADERS)


def fresh_ids():
    return {"ActivityId": str(uuid.uuid4()), "RequestId": str(uuid.uuid4())}


def discover_columns():
    """Try to list the entity's columns (and types) via conceptualschema."""
    url = f"{HOST}/public/reports/conceptualschema"
    try:
        r = session.post(url, json={"modelIds": [MODEL_ID]}, headers=fresh_ids(), timeout=60)
        r.raise_for_status()
        for schema in r.json().get("schemas", []):
            for ent in schema.get("schema", {}).get("Entities", []):
                if ent.get("Name") == ENTITY:
                    cols = [p["Name"] for p in ent.get("Properties", []) if "Column" in p]
                    if cols:
                        return cols
    except Exception as e:  # noqa: BLE001
        print(f"[warn] schema discovery failed ({e}); using COLUMNS fallback", file=sys.stderr)
    return COLUMNS


def build_body(columns, page_size, restart_tokens):
    select = [
        {
            "Column": {"Expression": {"SourceRef": {"Source": "_"}}, "Property": c},
            "Name": f"{ENTITY}.{c}",
        }
        for c in columns
    ]
    window = {"Count": page_size}
    if restart_tokens:
        window["RestartTokens"] = restart_tokens
    command = {
        "SemanticQueryDataShapeCommand": {
            "Query": {
                "Version": 2,
                "From": [{"Name": "_", "Entity": ENTITY, "Type": 0}],
                "Select": select,
            },
            "Binding": {
                "Primary": {"Groupings": [{"Projections": list(range(len(columns)))}]},
                "DataReduction": {"DataVolume": 4, "Primary": {"Window": window}},
                "Version": 1,
            },
            "ExecutionMetricsKind": 1,
        }
    }
    return {
        "version": "1.0.0",
        "queries": [{
            "Query": {"Commands": [command]},
            "QueryId": "",
            "ApplicationContext": {
                "DatasetId": DATASET_ID,
                "Sources": [{"ReportId": REPORT_ID}],
            },
        }],
        "cancelQueries": [],
        "modelId": MODEL_ID,
    }


def decode_ds(ds, ncols, types):
    """Decode one DS block of the compressed DSR format into a list of row lists."""
    dicts = ds.get("ValueDicts", {})
    rows, prev, schema = [], [None] * ncols, None
    for ph in ds.get("PH", []):
        for row in ph.get("DM0", []):
            if "S" in row:
                schema = row["S"]
            values = iter(row.get("C", []))
            rep, nul = row.get("R", 0), row.get("\u00d8", 0)
            cur = []
            for i in range(ncols):
                bit = 1 << i
                if rep & bit:
                    v = prev[i]
                elif nul & bit:
                    v = None
                else:
                    v = next(values, None)
                    dn = schema[i].get("DN") if schema else None
                    if dn is not None and isinstance(v, int):
                        v = dicts[dn][v]
                    # T=7 is DateTime, sent as epoch milliseconds
                    if schema and schema[i].get("T") == 7 and isinstance(v, (int, float)):
                        v = dt.datetime.fromtimestamp(v / 1000, dt.timezone.utc).isoformat()
                cur.append(v)
            prev = cur
            rows.append(cur)
    return rows


def fetch_all(columns, page_size):
    url = f"{HOST}/public/reports/querydata?synchronous=true"
    all_rows, tokens, page = [], None, 0
    while True:
        page += 1
        r = session.post(url, json=build_body(columns, page_size, tokens),
                         headers=fresh_ids(), timeout=120)
        r.raise_for_status()
        result = r.json()["results"][0]["result"]["data"]
        ds = result["dsr"]["DS"][0]
        if "odata.error" in str(result)[:200] or "DataShapes" in ds.get("Msg", [{}])[0:1].__repr__():
            raise RuntimeError(f"Query error: {result}")
        rows = decode_ds(ds, len(columns), None)
        all_rows.extend(rows)
        print(f"page {page}: +{len(rows)} rows (total {len(all_rows)})")

        # IC = "IsComplete": no more data to fetch. RT = restart tokens for next page.
        if ds.get("IC") or not ds.get("RT"):
            break
        tokens = ds["RT"]
        time.sleep(0.5)  # be polite
    return all_rows

In [ ]:
# 3. Run the scrape (takes a minute or so)
OUT = "incidents.csv"
PAGE_SIZE = 10000          # rows per request; Power BI caps this at about 30000
PREVIOUS = None            # optional: path of last week's CSV (upload it in the left panel) to see what changed

def run_scrape(out=OUT, page_size=PAGE_SIZE, tries=3):
    last = None
    for attempt in range(1, tries + 1):
        try:
            columns = discover_columns()
            print(f"Columns ({len(columns)})")
            rows = fetch_all(columns, page_size)
            with open(out, "w", newline="", encoding="utf-8") as f:
                w = csv.writer(f)
                w.writerow(columns)
                w.writerows(rows)
            print(f"Wrote {len(rows)} rows to {out}")
            return out
        except Exception as e:  # noqa: BLE001
            last = e
            print(f"[attempt {attempt}/{tries}] failed: {e}")
            time.sleep(5 * attempt)
    raise RuntimeError(f"Scrape failed after {tries} attempts: {last}")

run_scrape()

In [ ]:
# 4. Check the file before you upload it
import pandas as pd

REQUIRED = [
    "Governorate", "Residentialarea", "IncidentType", "NumberofDisplacedHHs", "Displaced people",
    "NumberofAffectedHHs", "Affected people", " Demolished structures", "Donor-funded structures",
    "Date of incident", "Area", "Reasonfordemolitioneviction (groups)", "Whocarriedoutthedemolitioneviction",
    "JV", "x", "y", "Classification (groups)", "Affected_Boys", "Affected_Girls", "Affected_Men", "Affected_Women",
    "PsychoNeeded_Ind", "Health", "WASH ", "Food", "Shelter ", "Livelihood ", "Legal aid", "Education",
    "ProtectivePresence_Ind", "Reg. refugee", "Classification", "Year", "PsychosocialNeed", "WASHNeed", "LivelihoodNeed",
    "Inhabited residential", "Uninhabited residential", "Agriculture", "WASH", "Infrastructure", "Livelihood", "Other",
]

def validate(path=OUT, previous=PREVIOUS):
    df = pd.read_csv(path)
    problems = []
    missing = [c for c in REQUIRED if c not in df.columns]
    if missing:
        problems.append(f"missing columns the page needs: {missing}")
    if len(df) < 4000:
        problems.append(f"only {len(df)} rows (expected 5,000+); the scrape may be partial")
    latest = None
    if "Date of incident" in df.columns:
        latest = df["Date of incident"].astype(str).str[:10].max()
    print(f"Rows: {len(df):,}   Columns: {df.shape[1]}   Latest incident: {latest}")
    if previous and os.path.exists(previous):
        old = pd.read_csv(previous)
        print(f"Previous file: {len(old):,} rows, so {len(df) - len(old):+,} rows")
        if len(df) < 0.9 * len(old):
            problems.append("far fewer rows than the previous file")
    if problems:
        print("\nDO NOT UPLOAD. Problems found:")
        for p in problems:
            print(" -", p)
    else:
        print("\nOK: this file looks complete. Upload it to your repo.")
    return not problems

ok = validate()

In [ ]:
# 5. Download to your computer
try:
    from google.colab import files
    files.download(OUT)
except ImportError:
    print(f"Not running in Colab; the file is at {os.path.abspath(OUT)}")